# 3.1 Missing data and imputation

We work through one dataset for the whole week: a survey of 654 students, with 11
Agree-Disagree items, some demographics and a few numeric questions. It is synthetic, but
every problem in it is one that turns up in real survey data.

Today: getting from a raw export to numbers you can trust, and deciding what to do about
the gaps. In 3.2 we transform and encode the result. In the 3.3 lab you do the same work
on datasets nobody built for teaching.

## The questionnaire

Five-point scale: Strongly disagree, Disagree, Neither agree nor disagree, Agree,
Strongly agree.

**Engagement with the course**

| Item | Wording | |
|---|---|---|
| `q1_lectures_useful` | I find the lectures useful | |
| `q2_keep_up_with_reading` | I keep up with the reading | |
| `q3_often_put_off_work` | I often put off my coursework | **reverse worded** |
| `q4_enjoy_problem_sets` | I enjoy working through problem sets | |
| `q5_ask_questions_in_class` | I ask questions when I do not understand | |

**Pressure and wellbeing**

| Item | Wording | |
|---|---|---|
| `q6_worry_about_deadlines` | I worry about deadlines | |
| `q7_sleep_well` | I sleep well during term | **reverse worded** |
| `q8_workload_manageable` | My workload is manageable | **reverse worded** |
| `q9_feel_under_pressure` | I feel under pressure most weeks | |
| `q10_time_for_myself` | I have enough time for myself | **reverse worded** |

`q11_attention_check` asked respondents to select "Disagree".

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

survey = pd.read_csv("data/student-survey-2026.csv")
survey.shape

FileNotFoundError: [Errno 2] No such file or directory: 'data/student-survey-2026.csv'

## 1. Look before you touch

Three lines, always, before anything else.

In [ ]:
survey.head()

In [ ]:
survey.dtypes

In [ ]:
survey.isna().sum()

Two things already:

- `age` is an integer, which means whatever is in it survived as a number. Not all numbers are appropriate ages.
- `hours_study_per_week` is `object`, which means at least one person typed words into a  numeric box.

And one thing `isna()` cannot tell you is what unique values a variable assumes when it is not a `NaN`:

In [ ]:
survey["q1_lectures_useful"].value_counts(dropna=False)

Five possible answers, many alternative spellings. Missing values are the easy part of a survey export.

## 2. One row per person

Before counting anything, check the unit of analysis is what you think it is.

In [ ]:
print("exact duplicate rows:", survey.duplicated().sum())
print("repeated respondent ids:", survey["respondent_id"].duplicated().sum())

In [ ]:
survey = survey.drop_duplicates(subset="respondent_id", keep="first").reset_index(drop=True)
survey.shape

Fourteen people submitted the form twice. Keeping both would weight them double in
every number you report afterwards.

## 3. Who was not paying attention

Two checks, and both have to happen **before** any recoding because recoding destroys the
evidence.

In [ ]:
items = [c for c in survey.columns if c.startswith("q") and "attention" not in c]

def tidy_text(s):
    return s.astype("string").str.strip().str.lower()

failed = tidy_text(survey["q11_attention_check"]).ne("disagree")
failed.sum()

The attention check asked for "Disagree". Note the `strip()` and `lower()`: without them
you would count `"Disagree "` as a failure.

Straight-lining is the second check: the same answer all the way down the grid.

In [ ]:
flat = survey[items].apply(tidy_text).nunique(axis=1).eq(1)
flat.sum()

This only works now. After reverse coding, a row of identical answers becomes a row of
alternating numbers and the pattern is invisible.

In [ ]:
drop = failed | flat
print(f"dropping {drop.sum()} of {len(survey)} ({drop.mean():.1%})")

survey = survey.loc[~drop].reset_index(drop=True)
survey.shape

Write that percentage down as you need to report it in the methods section of your report. A reader who is not told how many respondents were removed has no way to judge the rest.

## 4. Tidy the demographics

Free-text boxes and numeric codes, both of which need deciding before they can be grouped
by.

In [ ]:
survey["gender"].value_counts(dropna=False)

Nine spellings of four answers, plus blanks. Map them, and keep "Prefer not to say" as a
category: it is an answer, not a gap.

In [ ]:
gender_map = {"female": "Female", "f": "Female", "woman": "Female",
              "male": "Male", "m": "Male", "man": "Male",
              "non-binary": "Non-binary", "non binary": "Non-binary",
              "prefer not to say": "Prefer not to say"}

survey["gender"] = tidy_text(survey["gender"]).map(gender_map).fillna("Not given")
survey["gender"].value_counts()

The blanks became "Not given" rather than `NaN`. For a categorical column the two options are the most frequent category or an explicit "we do not know". The second keeps the fact explicit

In [ ]:
survey["year_of_study"].value_counts()

Eight spellings of three years, and the order matters, so it becomes an **ordered**
category rather than free text.

In [ ]:
year_map = {"first year": ...,  #TODO
            "1st year": ..., 
            "year 1": ...,
            "second year": ..., 
            "2nd year": ...,
            "third year": ..., 
            "3rd yr": ..., 
            "final year": ...}

survey["year_of_study"] = pd.Categorical(tidy_text(survey["year_of_study"]).map(year_map),
                                         categories=["First", "Second", "Third"], ordered=True)
survey["programme"] = survey["programme"].str.strip().str.title()

survey["year_of_study"].value_counts()

### Codes that look like numbers

In [ ]:
survey["age"].describe().round(1)

A maximum of 999 and a minimum of 0. Neither is an age:

- **999** is the form's code for "prefer not to answer"
- **0** is an empty box submitted as a number
- **12** is a typo

None of them are outliers to be capped. They are not measurements at all, so
we can set them as missing.

In [ ]:
survey["age"] = survey["age"].where(survey["age"].between(16, 80))
print("now missing:", survey["age"].isna().sum())
survey["age"].describe().round(1)

## 5. Words to numbers

The scale is ordered, so the mapping is not arbitrary: 5 is always the "most agree" end.

In [ ]:
scale = {"strongly disagree": 1, "disagree": 2, "neither agree nor disagree": 3,
         "agree": 4, "strongly agree": 5}

text_answers = survey[items].apply(tidy_text)
numeric = text_answers.apply(lambda s: s.map(scale))
numeric.head()

### Check what your own cleaning threw away

`.map()` is silent. Anything not in the dictionary becomes `NaN`, and it looks exactly like
someone who skipped the question. Always count both sides.

In [ ]:
before = text_answers.notna().sum().sum()
after = numeric.notna().sum().sum()
print("text answers:", before, " numbers:", after, " lost:", before - after)

In [ ]:
unmapped = pd.unique(text_answers.values.ravel())
[u for u in unmapped if pd.notna(u) and u not in scale]

`prefer not to say` is the only thing that did not map, and that is deliberate: it tells you the person declined. If anything else had appeared here, it would be a spelling we missed, and the fix would be the dictionary, not the data.

In [ ]:
survey[items] = numeric

## 6. Reverse coding

Four items are worded the other way round. Agreeing with "I often put off my coursework"
means *less* engagement.

Check the problem before you fix it:

In [ ]:
survey[["q1_lectures_useful", "q3_often_put_off_work"]].corr().iloc[0, 1].round(2)

Negative. Two items that are supposed to measure the same thing should move together so something is wrong.

**Have a think first.** Two items that should measure the same thing correlate at −0.58.
What has gone wrong, and how would you fix it?

<details>
<summary>Hint</summary>

Read the wording of both items out loud. Does agreeing with them mean the same
thing in each case?

</details>

<details>
<summary>Answer</summary>

`q3` is worded the other way round, so agreeing means *less* engagement. Reverse it
so that 5 always means more of what you are measuring:

```python
survey[reverse] = 6 - survey[reverse]
```

On a 1 to 5 scale it is `6 - x`. What would it be in a seven-point scale?

</details>

In [ ]:
reverse = ["q3_often_put_off_work", "q7_sleep_well",
           "q8_workload_manageable", "q10_time_for_myself"]

# Reverse the scale for these items
# TODO

survey[["q1_lectures_useful", "q3_often_put_off_work"]].corr().iloc[0, 1].round(2)

`6 - x` works because the scale runs 1 to 5, so 1 becomes 5 and 5 becomes 1. 

Now every engagement item should correlate positively with every other one:

In [ ]:
engagement_items = [c for c in items if c[:3] in ("q1_", "q2_", "q3_", "q4_", "q5_")]
pressure_items = [c for c in items if c not in engagement_items]

survey[engagement_items].corr().round(2)

<details>
<summary>Try comprehension yourself</summary>
Print the following items with a list comprehension:

```python
animals = ["cat", "dog", "elephant", "penguin"]
```
</details>

In [ ]:
(survey[items].isna().mean() * 100).round(1)

Roughly 3% on most items, which is people skipping a question, with no pattern to it. That is MCAR, and it is the benign case.

### MAR: sleep hours

In [ ]:
survey.groupby("year_of_study")["sleep_hours"].apply(lambda s: s.isna().mean().round(3))

First years skip it five times as often as everyone else. That is MAR: the missingness
depends on `year_of_study`, which we have, so we can condition on it.

### MNAR: monthly spending

In [ ]:
print("missing:", survey["monthly_spend_gbp"].isna().mean().round(3))

fig, ax = plt.subplots(figsize=(7, 3.2))
survey["monthly_spend_gbp"].plot.hist(bins=40, ax=ax)
ax.set_xlabel("monthly spend (GBP)")
plt.show()

About 11% missing, and the right-hand tail looks thin for a spending distribution. If the people who spend most are the ones who skip the question, the missingness depends on the value itself, which is MNAR.

**You cannot prove something is MNAR from the data**, because the evidence is exactly what is absent.
You argue it from how the question was asked, and you say in your write-up that you are
assuming it.

## 8. What each strategy does to your data

Arguments about imputation are usually tested by having a ground truth that you can recover. Here is the approach: take a column with no missing values, delete some of it on purpose in a way you control, and compare what each method recovers.

We use `grade_last_term`, and we make it **MAR**: students who are less engaged are less willing to report their mark.

In [ ]:
complete = survey.dropna(subset=["grade_last_term"]).copy()
complete["engagement"] = complete[engagement_items].mean(axis=1)

rng = np.random.default_rng(0)
truth = complete["grade_last_term"]

p_missing = np.where(complete["engagement"] < 3, 0.5, 0.05)
mask = rng.random(len(complete)) < p_missing
observed = truth.where(~mask)

print(f"deleted {mask.mean():.0%} of marks")

In [ ]:
low = complete["engagement"] < 3

filled = pd.DataFrame({
    "complete case": observed.dropna(),
    "mean": observed.fillna(observed.mean()),
    "median": observed.fillna(observed.median()),
    "group mean": observed.fillna(complete.groupby(low)["grade_last_term"].transform("mean")),
})

pd.DataFrame({"mean": [truth.mean()] + [filled[c].mean() for c in filled],
              "std": [truth.std()] + [filled[c].std() for c in filled]},
             index=["truth"] + list(filled.columns)).round(2)

Read it carefully:

- **Complete case** and **mean** filling are both biased upwards. The marks that went
  missing were the lower ones, so the average of what is left is too high, and filling
  with that average just spreads the same wrong number around.
- **Mean** filling also shrinks the standard deviation, because every filled row sits
  exactly on the mean. Any confidence interval you compute afterwards is too narrow.
- **Group mean** conditions on the column that caused the missingness, and lands closest.

That is what "MAR is fixable" means in practice: use what you know about why it is
missing.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
truth.plot.hist(bins=40, alpha=0.5, ax=ax, label="truth")
filled["mean"].plot.hist(bins=40, alpha=0.5, ax=ax, label="mean imputed")
ax.legend()
ax.set_xlabel("grade last term")
plt.show()

<details>
<summary>Why there is a spike? Give your answer before you reveal the answer </summary>

The spike are all the missing values imputed uisng the mean. Anyone who sees this histogram knows where you filled, and roughly how many missing values you had
</details>


## 9. Model-based imputation

Instead of one number for the whole column, predict each missing value from the other columns. This is worth doing when the other columns actually know something.

`sleep_hours` is a good candidate: it is missing MAR, and the pressure items are about sleep, workload and time, which all are plausible predictors of `sleep_hours`.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

X = survey[pressure_items + ["sleep_hours"]]

imputed = pd.DataFrame(IterativeImputer(random_state=0).fit_transform(X),
                       columns=X.columns, index=survey.index)

was_missing = survey["sleep_hours"].isna()

print("observed   mean %.2f  std %.2f" % (survey["sleep_hours"].mean(), survey["sleep_hours"].std()))
print("model fill mean %.2f  std %.2f" % (imputed.loc[was_missing, "sleep_hours"].mean(),
                                          imputed.loc[was_missing, "sleep_hours"].std()))

### Always check the invented values are possible

In [ ]:
imputed.loc[was_missing, "sleep_hours"].agg(["min", "max"]).round(2)

Between about 6 and 8 hours: plausible, because the model had columns that genuinely predict sleep. Nothing in the method guarantees this. The same imputer, given columns that know nothing about the target, will happily produce negative prices or ages of 300. You have to look.

Two things to notice:

- the filled values depend on a model you now have to describe
- every standard error afterwards is too small, because the analysis treats invented
  numbers as if they were measured

Keep the record of which rows you filled:

In [ ]:
survey["sleep_hours_imputed"] = was_missing
survey["sleep_hours"] = imputed["sleep_hours"]
survey["sleep_hours_imputed"].sum()

## 10. Missing items inside a scale

A scale score is the mean of its items, but only if enough items were answered. Two answers out of five will give you an unreliable score.

This is where your intuition is important. State a rule, apply it, and report it. For example: at least 4 of the 5 items need to be answered in order to proceed.

In [ ]:
def scale_score(frame, min_answered):
    score = frame.mean(axis=1)
    return score.where(frame.count(axis=1) >= min_answered)

survey["engagement"] = scale_score(survey[engagement_items], 4)
survey["pressure"] = scale_score(survey[pressure_items], 4)

survey[["engagement", "pressure"]].isna().sum()

`mean(axis=1)` skips missing values silently, which is why we need to say `count(axis=1)`. Without it, a respondent who answered one item gets a confident-looking score based on that one item.

Compare the alternatives before settling:

In [ ]:
comparison = pd.DataFrame({
    "person mean, 4 of 5": survey["engagement"],
    "item median filled": survey[engagement_items].fillna(survey[engagement_items].median()).mean(axis=1),
    "complete items only": survey[engagement_items].dropna().mean(axis=1),
})

pd.DataFrame({"mean": comparison.mean().round(3),
              "std": comparison.std().round(3),
              "n": comparison.count()})

All three give nearly the same mean here, and differ in how many respondents survive.
That is the usual situation: the choice matters less than saying which one you made.

## 11. Save it for 3.2

In [ ]:
survey.to_csv("data/survey-scored.csv", index=False)
survey.shape

## Check before you move on

1. Why do the attention check and straight-lining checks have to happen before reverse
   coding?
2. Mean imputation leaves the mean of a column unchanged. What does it do to the standard
   deviation, and why does that matter for anything you report afterwards?
3. `sleep_hours` is missing five times as often for first years. Which mechanism is that,
   and what does it let you do that MNAR would not?
4. A respondent answered 2 of the 5 engagement items. What is their engagement score under
   our rule, and why not just average the two?